# IAN 603 · Module 5
# Reasoning with SQL Logic

SQL logic is not mainly about memorizing operators.

It is about translating a real rule into conditions that behave correctly for:

- normal values,
- boundary values,
- missing values,
- blank text,
- TRUE / FALSE / UNKNOWN.

A useful workflow is:

```text
State the rule in plain language
          ↓
Identify edge cases
          ↓
Translate the rule to SQL
          ↓
Check TRUE / FALSE / UNKNOWN
          ↓
Test boundaries and NULLs
          ↓
Validate the result
```

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module5.html


## What you will learn

By the end of this notebook, you should be able to:

1. Combine conditions with `AND`, `OR`, and `NOT`.
2. Explain SQL operator precedence and why parentheses matter.
3. Explain SQL three-valued logic: TRUE, FALSE, and UNKNOWN.
4. Explain why `WHERE` keeps only TRUE rows.
5. Test NULL correctly with `IS NULL`.
6. Distinguish NULL, blank text, FALSE, and zero.
7. Use `IS TRUE`, `IS FALSE`, and `IS UNKNOWN`.
8. Use `IS DISTINCT FROM` for NULL-safe comparisons.
9. Normalize blank text with `NULLIF` and `BTRIM`.
10. Build fallback logic with `COALESCE`.
11. Avoid division-by-zero with `NULLIF`.
12. Use searched and simple `CASE`.
13. Order `CASE` branches correctly.
14. Recognize the `NOT IN (..., NULL)` trap.
15. Build and validate practical decision rules.


## 0. Install required Python packages

Run this cell if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** The helper is downloaded automatically when needed.

If you want your own copy:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)


In [ ]:
from pathlib import Path
import urllib.request
from sqlalchemy import text

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password and click **Connect**.


In [ ]:
db.show_connection_panel()


### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 3. Check the connection

After clicking **Connect**, run this cell once.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Meet the Edge Cases

This module intentionally includes awkward values.

The point is not to make the data look perfect. The point is to learn how SQL logic behaves when data is imperfect.

Examples include:

- `NULL` email
- blank email `''`
- whitespace-only email
- `NULL` consent
- zero bill
- `NULL` bill
- `NULL` payment status

These cases are where weak Boolean rules usually fail.


## 4. Create or reset the Module 5 teaching database


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 5\n-- Reasoning with SQL Logic\n-- ============================================================\n-- Companion file for s_module5.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- AND, OR, NOT, operator precedence, three-valued logic,\n-- NULL handling, IS TRUE / FALSE / UNKNOWN, IS DISTINCT FROM,\n-- COALESCE, NULLIF, CASE, and applied decision rules.\n--\n-- Important: WHERE keeps only TRUE. FALSE and UNKNOWN rows are\n-- both filtered out unless the rule explicitly handles them.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP VIEW IF EXISTS patient_case CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS patient_billing CASCADE;\nDROP TABLE IF EXISTS diagnostics CASCADE;\nDROP TABLE IF EXISTS patient_info CASCADE;\n\nCREATE TABLE patient_info (\n    patient_id integer PRIMARY KEY,\n    patient_name varchar(100) NOT NULL,\n    age integer NOT NULL CHECK (age BETWEEN 0 AND 120),\n    email_address varchar(150),\n    mobile_phone varchar(30),\n    home_phone varchar(30),\n    active boolean NOT NULL DEFAULT true,\n    consent_to_contact boolean,\n    preferred_language varchar(40)\n);\n\nINSERT INTO patient_info\n    (patient_id, patient_name, age, email_address, mobile_phone, home_phone, active, consent_to_contact, preferred_language)\nVALUES\n    (101, 'John Doe', 65, 'john.doe@email.com', '336-555-0101', NULL, true, true, 'English'),\n    (102, 'Jane Smith', 32, 'jane.smith@email.com', '336-555-0102', NULL, true, false, 'English'),\n    (103, 'Robert Lee', 48, NULL, NULL, '336-555-0103', true, true, 'English'),\n    (104, 'Lisa Ray', 71, '', NULL, '336-555-0104', true, false, 'English'),\n    (105, 'Peter Chen', 25, 'peter.chen@email.com', '336-555-0105', NULL, false, true, 'Mandarin'),\n    (106, 'Susan Gray', 58, 'susan.gray@email.com', NULL, NULL, true, NULL, NULL),\n    (107, 'Maria Torres', 44, '   ', '336-555-0107', NULL, true, true, 'Spanish'),\n    (108, 'Andre Green', 16, NULL, NULL, NULL, true, false, 'English');\n\nCREATE TABLE diagnostics (\n    patient_id integer PRIMARY KEY REFERENCES patient_info(patient_id),\n    diagnosis varchar(100),\n    severity varchar(20) CHECK (severity IN ('Low', 'Moderate', 'High')),\n    confirmed boolean\n);\nINSERT INTO diagnostics (patient_id, diagnosis, severity, confirmed) VALUES\n    (101, 'Pneumonia', 'High', true), (102, 'Fever', 'Moderate', true),\n    (103, 'Kidney Disease', 'High', true), (104, 'Diabetes', 'High', true),\n    (105, 'Sprained Ankle', 'Low', true), (106, 'Kidney Disease', 'High', NULL),\n    (107, 'Asthma', 'Moderate', true), (108, 'Flu', 'Moderate', false);\n\nCREATE TABLE patient_billing (\n    patient_id integer PRIMARY KEY REFERENCES patient_info(patient_id),\n    bill_amount numeric(10,2) CHECK (bill_amount IS NULL OR bill_amount >= 0),\n    amount_paid numeric(10,2) CHECK (amount_paid IS NULL OR amount_paid >= 0),\n    payment_status varchar(30)\n);\nINSERT INTO patient_billing (patient_id, bill_amount, amount_paid, payment_status) VALUES\n    (101, 12000.00, 4000.00, 'Open'), (102, 7500.00, 7500.00, 'Paid'),\n    (103, NULL, 0.00, 'Pending Review'), (104, 9800.00, NULL, 'Open'),\n    (105, 850.00, 850.00, 'Paid'), (106, 5200.00, 1000.00, NULL),\n    (107, 0.00, 0.00, 'No Charge'), (108, 2300.00, 0.00, 'Open');\n\nCREATE TABLE admissions (\n    admission_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patient_info(patient_id),\n    admission_date date NOT NULL,\n    discharge_date date,\n    diagnosis varchar(100) NOT NULL\n);\nINSERT INTO admissions (admission_id, patient_id, admission_date, discharge_date, diagnosis) VALUES\n    (201, 101, '2026-07-15', '2026-07-21', 'Pneumonia'),\n    (202, 102, '2026-07-25', '2026-07-27', 'Fever'),\n    (203, 103, '2026-08-01', NULL, 'Kidney Disease'),\n    (204, 104, '2026-06-10', '2026-06-22', 'Diabetes'),\n    (205, 105, '2026-08-05', '2026-08-06', 'Sprained Ankle'),\n    (206, 106, '2026-07-30', '2026-08-04', 'Kidney Disease'),\n    (207, 107, '2026-08-08', NULL, 'Asthma'),\n    (208, 108, '2026-08-09', '2026-08-11', 'Flu');\n\nCREATE VIEW patient_case AS\nSELECT p.patient_id, p.patient_name, p.age, p.email_address, p.mobile_phone, p.home_phone,\n       p.active, p.consent_to_contact, p.preferred_language,\n       d.diagnosis, d.severity, d.confirmed,\n       b.bill_amount, b.amount_paid, b.payment_status\nFROM patient_info AS p\nLEFT JOIN diagnostics AS d USING (patient_id)\nLEFT JOIN patient_billing AS b USING (patient_id);"

execute_script(setup_sql)


## 5. Inspect the main patient case view


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    age,
    email_address,
    active,
    consent_to_contact,
    preferred_language,
    diagnosis,
    severity,
    confirmed,
    bill_amount,
    amount_paid,
    payment_status
FROM patient_case
ORDER BY patient_id;
""", title="Patient cases")


## 6. Identify the edge cases before writing rules

A strong SQL rule begins with inspection.

Ask:

- Which columns contain NULL?
- Which text columns contain blanks?
- Which Boolean columns can be UNKNOWN?
- Which numeric columns contain zero?
- Which values sit exactly on a threshold?


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    email_address,
    consent_to_contact,
    bill_amount,
    amount_paid,
    payment_status
FROM patient_case
WHERE email_address IS NULL
   OR BTRIM(COALESCE(email_address, '')) = ''
   OR consent_to_contact IS NULL
   OR bill_amount IS NULL
   OR bill_amount = 0
   OR payment_status IS NULL
ORDER BY patient_id;
""", title="Rows with important edge cases")


# Part II · AND, OR, and NOT

Start with the business rule in plain language.

Example:

> Keep patients who are active, at least 40 years old, and have High severity.

That becomes three conditions joined with `AND`.

```text
active
  AND age >= 40
  AND severity = 'High'
```

All three conditions must evaluate to TRUE for the row to remain.


## 7. AND: every condition must be TRUE


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    age,
    diagnosis,
    severity
FROM patient_case
WHERE active IS TRUE
  AND age >= 40
  AND severity = 'High'
ORDER BY patient_id;
""", title="Active, age 40+, High severity")


## 8. OR: at least one condition may be TRUE

Suppose the rule is:

> Keep respiratory diagnoses: Pneumonia, Flu, or Asthma.

We could write several `OR` conditions, but `IN` is clearer when comparing one column to several exact values.


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    diagnosis
FROM patient_case
WHERE diagnosis IN ('Pneumonia', 'Flu', 'Asthma')
ORDER BY patient_id;
""", title="Selected respiratory diagnoses")


## 9. NOT: reverse a condition

This rule means:

> Keep active patients whose diagnosis is not Sprained Ankle or Flu.


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    diagnosis,
    active
FROM patient_case
WHERE active IS TRUE
  AND NOT (diagnosis IN ('Sprained Ankle', 'Flu'))
ORDER BY patient_id;
""", title="Active patients excluding two diagnoses")


# Part III · Operator Precedence and Parentheses

SQL does not evaluate `AND` and `OR` equally.

A useful rule to remember:

> **AND is evaluated before OR.**

So these two rules are different.

### Rule A

```sql
age >= 65
OR diagnosis = 'Kidney Disease'
AND consent_to_contact IS TRUE
```

This behaves like:

```text
age >= 65
OR (kidney disease AND consent)
```

### Rule B

```sql
(age >= 65 OR diagnosis = 'Kidney Disease')
AND consent_to_contact IS TRUE
```

Now consent applies to the entire age/diagnosis condition.


## 10. Compare the two rules side by side


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    age,
    diagnosis,
    consent_to_contact,
    (
        age >= 65
        OR diagnosis = 'Kidney Disease'
           AND consent_to_contact IS TRUE
    ) AS rule_a,
    (
        (age >= 65 OR diagnosis = 'Kidney Disease')
        AND consent_to_contact IS TRUE
    ) AS rule_b
FROM patient_case
ORDER BY patient_id;
""", title="Operator precedence comparison")


### Practical habit

Whenever a rule mixes `AND` and `OR`, add parentheses that show the intended business logic.

Do not rely on the reader remembering precedence.


# Part IV · SQL Has Three Truth Values

SQL Boolean logic has:

- TRUE
- FALSE
- UNKNOWN

UNKNOWN often appears when a comparison involves `NULL`.

This matters because:

> **WHERE keeps only rows where the condition is TRUE.**

FALSE and UNKNOWN are both filtered out.


## 11. Build a TRUE / FALSE / UNKNOWN truth table


In [ ]:
display_query("""
WITH bools(label, value, sort_order) AS (
    VALUES
        ('TRUE', TRUE, 1),
        ('FALSE', FALSE, 2),
        ('UNKNOWN', NULL::boolean, 3)
)
SELECT
    a.label AS a,
    b.label AS b,
    COALESCE((a.value AND b.value)::text, 'unknown') AS a_and_b,
    COALESCE((a.value OR b.value)::text, 'unknown') AS a_or_b
FROM bools AS a
CROSS JOIN bools AS b
ORDER BY a.sort_order, b.sort_order;
""", title="Three-valued logic")


### A few important cases

| A | B | A AND B | A OR B |
|---|---|---|---|
| TRUE | UNKNOWN | UNKNOWN | TRUE |
| FALSE | UNKNOWN | FALSE | UNKNOWN |
| UNKNOWN | UNKNOWN | UNKNOWN | UNKNOWN |

UNKNOWN is not the same as FALSE.


## 12. WHERE keeps only TRUE


In [ ]:
display_query("""
SELECT label, flag
FROM (
    VALUES
        ('TRUE row', TRUE),
        ('FALSE row', FALSE),
        ('UNKNOWN row', NULL::boolean)
) AS states(label, flag)
WHERE flag;
""", title="Only TRUE survives WHERE")


# Part V · Why = NULL Fails

A common beginner mistake is:

```sql
email_address = NULL
```

or:

```sql
email_address <> NULL
```

Neither returns TRUE.

Comparisons with `NULL` produce UNKNOWN.

Use:

```sql
IS NULL
```

or:

```sql
IS NOT NULL
```


## 13. See the problem directly


In [ ]:
display_query("""
SELECT
    patient_id,
    email_address,
    email_address = NULL AS equals_null,
    email_address <> NULL AS not_equals_null,
    email_address IS NULL AS is_null
FROM patient_info
ORDER BY patient_id;
""", title="NULL comparison behavior")


# Part VI · NULL, Blank Text, FALSE, and Zero Are Different

These values may all look like “nothing,” but they are not equivalent.

| Value | Meaning example |
|---|---|
| `NULL` | email not recorded |
| `''` | empty text stored |
| `'   '` | whitespace-only text |
| `FALSE` | explicit no |
| `0` | numeric zero |

Good logic keeps these meanings separate until there is a justified reason to normalize them.


## 14. Classify email states


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    email_address,
    CASE
        WHEN email_address IS NULL THEN 'NULL'
        WHEN email_address = '' THEN 'Empty string'
        WHEN BTRIM(email_address) = '' THEN 'Whitespace only'
        ELSE 'Recorded'
    END AS email_state
FROM patient_info
ORDER BY patient_id;
""", title="NULL vs blank text")


# Part VII · Boolean Predicates

A Boolean column can itself be TRUE, FALSE, or NULL.

Instead of writing:

```sql
consent_to_contact = TRUE
```

PostgreSQL supports explicit predicates:

- `IS TRUE`
- `IS FALSE`
- `IS UNKNOWN`

These are especially readable when NULL has a real meaning such as “not recorded.”


## 15. Separate yes, no, and not recorded


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    consent_to_contact,
    consent_to_contact IS TRUE AS explicitly_yes,
    consent_to_contact IS FALSE AS explicitly_no,
    consent_to_contact IS UNKNOWN AS not_recorded
FROM patient_info
ORDER BY patient_id;
""", title="Boolean predicates")


# Part VIII · NULL-Safe Comparison with IS DISTINCT FROM

Ordinary equality and inequality become UNKNOWN when NULL is involved.

PostgreSQL provides:

- `IS DISTINCT FROM`
- `IS NOT DISTINCT FROM`

These treat NULL as a comparable state.

That is useful when “different even when one side is missing” is the intended rule.


## 16. Compare billing amounts NULL-safely


In [ ]:
display_query("""
SELECT
    patient_id,
    bill_amount,
    amount_paid,
    bill_amount IS DISTINCT FROM amount_paid AS amounts_differ,
    bill_amount IS NOT DISTINCT FROM amount_paid AS amounts_match_null_safely
FROM patient_billing
ORDER BY patient_id;
""", title="NULL-safe comparison")


# Part IX · Normalize Blank Text with NULLIF

A common normalization pattern is:

```sql
NULLIF(BTRIM(email_address), '')
```

Read it in two steps:

1. `BTRIM(...)` removes surrounding whitespace.
2. `NULLIF(value, '')` converts an empty result to NULL.

Examples:

| raw value | normalized |
|---|---|
| `'a@x.com'` | `'a@x.com'` |
| `' a@x.com '` | `'a@x.com'` |
| `''` | NULL |
| `'   '` | NULL |
| NULL | NULL |


## 17. Normalize email values


In [ ]:
display_query("""
SELECT
    patient_id,
    email_address AS raw_email,
    NULLIF(BTRIM(email_address), '') AS normalized_email
FROM patient_info
ORDER BY patient_id;
""", title="Normalize blank email values")


# Part X · Build Fallback Logic with COALESCE

`COALESCE` returns the first non-NULL value.

A practical contact rule might be:

```text
clean email
   ↓ if missing
clean mobile
   ↓ if missing
clean home phone
   ↓ if missing
fallback text
```

This is much stronger than using raw values directly because blank strings are normalized first.


## 18. Build a preferred contact value


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    COALESCE(
        NULLIF(BTRIM(email_address), ''),
        NULLIF(BTRIM(mobile_phone), ''),
        NULLIF(BTRIM(home_phone), ''),
        'No contact provided'
    ) AS preferred_contact
FROM patient_info
ORDER BY patient_id;
""", title="Preferred contact")


### Why normalize before COALESCE?

Without `NULLIF(BTRIM(...), '')`, an empty string counts as a non-NULL value.

So `COALESCE` could incorrectly stop at a blank email instead of falling through to the phone number.


# Part XI · Use NULLIF to Prevent Division by Zero

Suppose we calculate:

```text
percent paid = amount_paid / bill_amount × 100
```

If `bill_amount = 0`, division would fail.

This pattern converts zero to NULL:

```sql
NULLIF(bill_amount, 0)
```

Then the division returns NULL instead of raising a division-by-zero error.


## 19. Safe percentage calculation


In [ ]:
display_query("""
SELECT
    patient_id,
    bill_amount,
    amount_paid,
    ROUND(
        100.0 * COALESCE(amount_paid, 0)
        / NULLIF(bill_amount, 0),
        1
    ) AS percent_paid
FROM patient_billing
ORDER BY patient_id;
""", title="Percent paid with a safe denominator")


# Part XII · CASE: Turn Rules into Categories

A searched `CASE` asks conditions from top to bottom.

```sql
CASE
    WHEN condition_1 THEN result_1
    WHEN condition_2 THEN result_2
    ELSE fallback
END
```

The first TRUE condition wins.

That means **branch order matters**.


## 20. Build billing tiers with searched CASE


In [ ]:
display_query("""
SELECT
    patient_id,
    bill_amount,
    CASE
        WHEN bill_amount IS NULL THEN 'Unknown'
        WHEN bill_amount = 0 THEN 'No Charge'
        WHEN bill_amount >= 10000 THEN 'High-Value'
        WHEN bill_amount >= 5000 THEN 'Standard'
        ELSE 'Low-Value'
    END AS billing_tier
FROM patient_billing
ORDER BY patient_id;
""", title="Billing tiers")


### Why this order?

Specific cases come first:

1. missing
2. exactly zero
3. high threshold
4. lower threshold
5. fallback

If the broad condition comes first, later branches may never be reached.


## 21. Simple CASE

Simple `CASE` compares one expression to several exact values.

It is useful when the logic is:

> if status equals X, return Y.


In [ ]:
display_query("""
SELECT
    patient_id,
    payment_status,
    CASE payment_status
        WHEN 'Paid' THEN 'Closed'
        WHEN 'No Charge' THEN 'Closed'
        WHEN 'Open' THEN 'Follow up'
        WHEN 'Pending Review' THEN 'Investigate'
        ELSE 'Status missing'
    END AS action
FROM patient_billing
ORDER BY patient_id;
""", title="Simple CASE for payment status")


# Part XIII · CASE Branch Order

Compare these two rules.

### Wrong order

```sql
WHEN age >= 18 THEN 'Adult'
WHEN age >= 65 THEN 'Senior'
```

A 71-year-old matches `age >= 18` first, so the Senior branch is never reached.

### Correct order

```sql
WHEN age >= 65 THEN 'Senior'
WHEN age >= 18 THEN 'Adult'
```

Move from specific / high-priority conditions toward broader conditions.


## 22. Compare wrong and correct branch order


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    age,
    CASE
        WHEN age >= 18 THEN 'Adult'
        WHEN age >= 65 THEN 'Senior'
        ELSE 'Minor'
    END AS wrong_order,
    CASE
        WHEN age >= 65 THEN 'Senior'
        WHEN age >= 18 THEN 'Adult'
        ELSE 'Minor'
    END AS correct_order
FROM patient_info
ORDER BY patient_id;
""", title="CASE branch order")


# Part XIV · Missing ELSE

If no `WHEN` condition matches and there is no `ELSE`, the result is NULL.

That may be useful if NULL intentionally means “unclassified.”

But it may also hide an incomplete rule.


## 23. See what happens without ELSE


In [ ]:
display_query("""
SELECT
    patient_id,
    diagnosis,
    CASE
        WHEN diagnosis IN ('Pneumonia', 'Flu', 'Asthma')
            THEN 'Respiratory'
    END AS diagnosis_group
FROM diagnostics
ORDER BY patient_id;
""", title="CASE without ELSE")


### Design question

Should unmatched diagnoses be:

- NULL?
- `'Other'`?
- `'Needs review'`?

There is no universal answer. Choose the meaning intentionally.


# Part XV · The NOT IN + NULL Trap

This is one of the most important SQL logic traps.

Consider:

```sql
payment_status NOT IN ('Paid', NULL)
```

The NULL inside the list introduces UNKNOWN comparisons.

The result may be no rows at all—even when many statuses are visibly not Paid.


## 24. Demonstrate the trap


In [ ]:
display_query("""
SELECT
    patient_id,
    payment_status
FROM patient_billing
WHERE payment_status NOT IN ('Paid', NULL)
ORDER BY patient_id;
""", title="NOT IN with NULL")


### Better question

If you mean:

> include any status that is not definitely Paid, including missing status,

use a NULL-safe predicate.


## 25. Ordinary inequality vs NULL-safe inequality


In [ ]:
display_query("""
SELECT
    patient_id,
    payment_status,
    payment_status <> 'Paid' AS ordinary_not_paid,
    payment_status IS DISTINCT FROM 'Paid' AS null_safe_not_paid
FROM patient_billing
ORDER BY patient_id;
""", title="Inequality comparison")


# Part XVI · Applied Rule 1: High-Risk Outreach

Now combine several concepts into one business rule.

Plain language:

> Keep active patients who explicitly consented to contact, have Diabetes or Kidney Disease, and do not have a usable email address.

Break it into pieces:

| Rule | SQL idea |
|---|---|
| active | `active IS TRUE` |
| explicit consent | `consent_to_contact IS TRUE` |
| diagnosis set | `IN (...)` |
| no usable email | normalize blank/whitespace to NULL |


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    diagnosis,
    consent_to_contact,
    email_address
FROM patient_case
WHERE active IS TRUE
  AND consent_to_contact IS TRUE
  AND diagnosis IN ('Diabetes', 'Kidney Disease')
  AND NULLIF(BTRIM(email_address), '') IS NULL
ORDER BY patient_id;
""", title="High-risk outreach rule")


# Part XVII · Applied Rule 2: Contact + Billing

A strong analytical query can combine normalized values and categories.

This query:

1. creates a preferred contact,
2. preserves missing information,
3. assigns a billing tier.


In [ ]:
display_query("""
SELECT
    patient_id,
    patient_name,
    COALESCE(
        NULLIF(BTRIM(email_address), ''),
        NULLIF(BTRIM(mobile_phone), ''),
        NULLIF(BTRIM(home_phone), ''),
        'No contact provided'
    ) AS preferred_contact,
    bill_amount,
    CASE
        WHEN bill_amount IS NULL THEN 'Needs review'
        WHEN bill_amount = 0 THEN 'No Charge'
        WHEN bill_amount >= 10000 THEN 'High-Value'
        WHEN bill_amount >= 5000 THEN 'Standard'
        ELSE 'Low-Value'
    END AS billing_tier
FROM patient_case
ORDER BY patient_id;
""", title="Normalized contact and billing tier")


# Part XVIII · Applied Rule 3: Length-of-Stay Categories

A classification rule should make its boundaries obvious.

Example:

| stay days | category |
|---:|---|
| less than 5 | Short |
| 5 through 10 | Medium |
| more than 10 | Long |

Boundary values deserve explicit testing.


## 26. Classify completed admissions


In [ ]:
display_query("""
SELECT
    admission_id,
    patient_id,
    admission_date,
    discharge_date,
    discharge_date - admission_date AS length_of_stay_days,
    CASE
        WHEN discharge_date - admission_date < 5 THEN 'Short'
        WHEN discharge_date - admission_date <= 10 THEN 'Medium'
        ELSE 'Long'
    END AS stay_category
FROM admissions
WHERE discharge_date IS NOT NULL
ORDER BY admission_id;
""", title="Length-of-stay categories")


## 27. Test boundary values directly

This small `VALUES` table lets us verify the rule independently from the patient data.


In [ ]:
display_query("""
SELECT
    stay_days,
    CASE
        WHEN stay_days < 5 THEN 'Short'
        WHEN stay_days <= 10 THEN 'Medium'
        ELSE 'Long'
    END AS stay_category
FROM (
    VALUES
        (4),
        (5),
        (10),
        (11)
) AS test_cases(stay_days)
ORDER BY stay_days;
""", title="Boundary tests")


# Part XIX · Reproducible Date Rules

Avoid rules based on an unstated “today.”

For teaching and reproducible analysis, use an explicit reference date.

Example:

```sql
DATE '2026-08-11'
```

Then the same query produces the same result whenever students run it.


## 28. A reproducible recent-admission rule


In [ ]:
display_query("""
SELECT
    a.admission_id,
    p.patient_name,
    p.age,
    a.admission_date,
    a.diagnosis,
    CASE
        WHEN a.diagnosis IN ('Pneumonia', 'Flu')
             AND p.age >= 60
            THEN 'High-Risk Respiratory'
        WHEN a.diagnosis IN ('Pneumonia', 'Flu')
            THEN 'Respiratory'
        ELSE 'Other'
    END AS diagnosis_category
FROM admissions AS a
JOIN patient_info AS p USING (patient_id)
WHERE a.admission_date
      BETWEEN DATE '2026-08-11' - 30
          AND DATE '2026-08-11'
  AND a.diagnosis IN ('Pneumonia', 'Flu')
ORDER BY a.admission_date;
""", title="Reproducible 30-day rule")


# Part XX · Validate Logic, Not Just Syntax

A query can run successfully and still encode the wrong rule.

Validation should target:

- NULL cases
- blank text
- TRUE / FALSE / UNKNOWN
- boundary values
- overlapping CASE branches
- unexpected categories
- row counts before and after filtering


## 29. Compare rule counts


In [ ]:
display_query("""
SELECT
    COUNT(*) AS all_patients,
    COUNT(*) FILTER (
        WHERE consent_to_contact IS TRUE
    ) AS explicit_yes,
    COUNT(*) FILTER (
        WHERE consent_to_contact IS FALSE
    ) AS explicit_no,
    COUNT(*) FILTER (
        WHERE consent_to_contact IS UNKNOWN
    ) AS not_recorded
FROM patient_info;
""", title="Boolean-state validation")


## 30. Validate contact normalization


In [ ]:
display_query("""
SELECT
    COUNT(*) AS patients,
    COUNT(*) FILTER (
        WHERE email_address IS NULL
    ) AS raw_null_email,
    COUNT(*) FILTER (
        WHERE NULLIF(BTRIM(email_address), '') IS NULL
    ) AS no_usable_email
FROM patient_info;
""", title="Raw NULL vs no usable email")


Notice that “raw NULL email” and “no usable email” are not necessarily the same count.

Normalization changes the **interpretation**, not just the formatting.


# Part XXI · Guided Practice

For each task:

1. Write the rule in plain language.
2. Identify possible NULL or blank cases.
3. Decide whether parentheses are needed.
4. Write the SQL.
5. Test edge cases.


## Practice 1 · Parentheses

Write a rule that requires explicit consent for this entire condition:

> age 65 or older OR Kidney Disease.

<details>
<summary>Hint</summary>

Put the age/diagnosis rule inside parentheses, then apply consent with `AND`.

</details>


In [ ]:
practice_1 = """
SELECT
    patient_id,
    patient_name,
    age,
    diagnosis,
    consent_to_contact
FROM patient_case
WHERE
    -- grouped age/diagnosis rule
    -- explicit consent
ORDER BY patient_id;
"""

# display_query(practice_1, title="Practice 1")


## Practice 2 · Preferred contact

Build:

```text
email → mobile → home phone → fallback
```

Treat blank and whitespace-only text as missing.


In [ ]:
practice_2 = """
SELECT
    patient_id,
    patient_name,
    COALESCE(
        -- normalized email,
        -- normalized mobile,
        -- normalized home,
        'No contact provided'
    ) AS preferred_contact
FROM patient_info
ORDER BY patient_id;
"""


## Practice 3 · Billing tiers

Create:

- Unknown
- No Charge
- High-Value
- Standard
- Low-Value

Think carefully about branch order.


In [ ]:
practice_3 = """
SELECT
    patient_id,
    bill_amount,
    CASE
        -- branches
    END AS billing_tier
FROM patient_billing
ORDER BY patient_id;
"""


## Practice 4 · NULL-safe status

Return rows whose payment status is not definitely `'Paid'`.

Include NULL status.


In [ ]:
practice_4 = """
SELECT
    patient_id,
    payment_status
FROM patient_billing
WHERE
    -- NULL-safe condition
ORDER BY patient_id;
"""


## Practice 5 · Length of stay

Classify completed admissions:

- Short: fewer than 5 days
- Medium: 5 through 10 days
- Long: more than 10 days

Test 4, 5, 10, and 11 days separately.


In [ ]:
practice_5 = """
SELECT
    admission_id,
    discharge_date - admission_date AS stay_days,
    CASE
        -- branches
    END AS stay_category
FROM admissions
WHERE discharge_date IS NOT NULL
ORDER BY admission_id;
"""


# Part XXII · Final Mini-Lab

Build an **outreach priority** result with:

- patient ID
- patient name
- preferred contact
- diagnosis
- consent status
- billing tier
- outreach priority

Suggested priority logic:

### Priority 1
Active + explicit consent + High severity + usable contact

### Priority 2
Active + explicit consent + Moderate severity + usable contact

### Not eligible
Anything else

Before coding, decide exactly what counts as a “usable contact.”


In [ ]:
final_query = """
SELECT
    patient_id,
    patient_name,
    -- preferred contact,
    diagnosis,
    consent_to_contact,
    -- billing tier,
    CASE
        -- outreach priority branches
    END AS outreach_priority
FROM patient_case
ORDER BY patient_id;
"""

# display_query(final_query, title="Final mini-lab")


# Quick Concept Check

### Why can `column = NULL` never be TRUE?
<details><summary>Show answer</summary>Comparisons with NULL produce UNKNOWN. Use `IS NULL` instead.</details>

### What does WHERE do with UNKNOWN?
<details><summary>Show answer</summary>It filters the row out. WHERE keeps only TRUE.</details>

### Why use parentheses when AND and OR are mixed?
<details><summary>Show answer</summary>To make the intended business rule explicit and avoid relying on precedence.</details>

### What does `NULLIF(BTRIM(email), '')` accomplish?
<details><summary>Show answer</summary>It trims whitespace and converts an empty result to NULL.</details>

### What does COALESCE return?
<details><summary>Show answer</summary>The first non-NULL expression.</details>

### Why can CASE branch order change the answer?
<details><summary>Show answer</summary>The first TRUE WHEN branch wins.</details>

### Why can `NOT IN ('Paid', NULL)` be dangerous?
<details><summary>Show answer</summary>The NULL introduces UNKNOWN comparisons and can cause rows you expect to keep to be filtered out.</details>

### What is a NULL-safe inequality in PostgreSQL?
<details><summary>Show answer</summary>`IS DISTINCT FROM`</details>


# Module 5 Summary

Strong SQL logic follows a reasoning process:

1. State the business rule clearly.
2. Identify NULL, blank, zero, FALSE, and boundary cases.
3. Use parentheses when `AND` and `OR` are mixed.
4. Remember that SQL has TRUE, FALSE, and UNKNOWN.
5. Remember that `WHERE` keeps only TRUE.
6. Normalize blank text before fallback logic.
7. Use `COALESCE` only after deciding what counts as missing.
8. Use `NULLIF` for normalization and safe denominators.
9. Order `CASE` branches from specific/high-priority to broader cases.
10. Test boundaries and NULLs explicitly.
11. Validate the meaning of the result, not just whether the query runs.

## Next
**Module 6: Joining Data in PostgreSQL**
